# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [3]:
%pip install iso3166

### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [4]:
%pip install --upgrade plotly

### Import Statements

In [5]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

### Notebook Presentation

In [6]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [7]:
df_data = pd.read_csv('mission_launches.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`?
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [8]:
print(f"Shape of df_data: {df_data.shape}")
print(f"Number of rows: {df_data.shape[0]}")
print(f"Number of columns: {df_data.shape[1]}")
print("\nColumn names:")
print(df_data.columns.tolist())
print("\nNaN values per column:")
print(df_data.isnull().sum())
print(f"\nNumber of duplicate rows: {df_data.duplicated().sum()}")

Shape of df_data: (4324, 9)
Number of rows: 4324
Number of columns: 9

Column names:
['Unnamed: 0.1', 'Unnamed: 0', 'Organisation', 'Location', 'Date', 'Detail', 'Rocket_Status', 'Price', 'Mission_Status']

NaN values per column:
Unnamed: 0.1         0
Unnamed: 0           0
Organisation         0
Location             0
Date                 0
Detail               0
Rocket_Status        0
Price             3360
Mission_Status       0
dtype: int64

Number of duplicate rows: 0


## Data Cleaning - Check for Missing Values and Duplicates

Consider removing columns containing junk data.

In [9]:
df_data = df_data.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])
print("Dropped 'Unnamed: 0.1' and 'Unnamed: 0' columns.")
print(f"New shape of df_data: {df_data.shape}")

Dropped 'Unnamed: 0.1' and 'Unnamed: 0' columns.
New shape of df_data: (4324, 7)


## Descriptive Statistics

In [10]:
print(df_data.describe(include='all'))

       Organisation                                    Location  \
count          4324                                        4324   
unique           56                                         137   
top       RVSN USSR  Site 31/6, Baikonur Cosmodrome, Kazakhstan   
freq           1777                                         235   

                              Date                               Detail  \
count                         4324                                 4324   
unique                        4319                                 4278   
top     Tue Aug 28, 1990 09:05 UTC  Cosmos-3MRB (65MRB) | BOR-5 Shuttle   
freq                             2                                    6   

        Rocket_Status  Price Mission_Status  
count            4324    964           4324  
unique              2     56              4  
top     StatusRetired  450.0        Success  
freq             3534    136           3879  


# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [11]:
launches_by_organisation = df_data['Organisation'].value_counts().reset_index()
launches_by_organisation.columns = ['Organisation', 'Number of Launches']

fig = px.bar(launches_by_organisation,
             x='Organisation',
             y='Number of Launches',
             title='Number of Space Mission Launches by Organisation',
             labels={'Organisation': 'Organisation', 'Number of Launches': 'Number of Launches'},
             color='Number of Launches',
             color_continuous_scale=px.colors.sequential.Plasma)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned?

In [12]:
rocket_status_counts = df_data['Rocket_Status'].value_counts()
print(rocket_status_counts)

Rocket_Status
StatusRetired    3534
StatusActive      790
Name: count, dtype: int64


# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [13]:
mission_status_counts = df_data['Mission_Status'].value_counts()
print(mission_status_counts)

Mission_Status
Success              3879
Failure               339
Partial Failure       102
Prelaunch Failure       4
Name: count, dtype: int64


# How Expensive are the Launches?

Create a histogram and visualise the distribution. The price column is given in USD millions (careful of missing values).

In [14]:
price_data = df_data.dropna(subset=['Price'])

fig = px.histogram(price_data,
                   x='Price',
                   nbins=50,
                   title='Distribution of Rocket Launch Prices (USD Millions)',
                   labels={'Price': 'Price (USD Millions)', 'count': 'Number of Launches'})

fig.update_layout(xaxis_title='Price (USD Millions)', yaxis_title='Number of Launches')
fig.show()

# Use a Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Experiment with [plotly's available colours](https://plotly.com/python/builtin-colorscales/). I quite like the sequential colour `matter` on this map.
* You'll need to extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

You'll need to use a 3 letter country code for each country. You might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


You can use the iso3166 package to convert the country names to Alpha3 format.

In [15]:
df_map = df_data.copy()

# Extract country from Location, taking the last part after the comma
df_map['Country'] = df_map['Location'].apply(lambda x: x.split(',')[-1].strip())

# Manual replacements for specific locations
country_replacements = {
    'Russia': 'Russian Federation',
    'New Mexico': 'USA',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'USA',
    'North Korea': 'Dem. People\'s Rep. Korea', # ISO name
    'South Korea': 'Republic of Korea', # ISO name
    'Iran': 'Iran (Islamic Republic of)', # ISO name
    'Kazakhstan': 'Kazakhstan' # Already correct, but good to have for clarity
}
df_map['Country'] = df_map['Country'].replace(country_replacements)

# Define a function to convert country names to ISO Alpha-3 codes
def get_iso_alpha3(country_name):
    try:
        # Some country names might need slight adjustments for iso3166
        if country_name == 'USA':
            return 'USA'
        elif country_name == 'Russian Federation':
            return 'RUS'
        elif country_name == 'China':
            return 'CHN'
        elif country_name == 'Iran (Islamic Republic of)':
            return 'IRN'
        elif country_name == 'Japan':
            return 'JPN'
        elif country_name == 'India':
            return 'IND'
        elif country_name == 'France':
            return 'FRA'
        elif country_name == 'Brazil':
            return 'BRA'
        elif country_name == 'New Zealand':
            return 'NZL'
        elif country_name == 'Australia':
            return 'AUS'
        elif country_name == 'Israel':
            return 'ISR'
        elif country_name == 'Dem. People\'s Rep. Korea': # North Korea
            return 'PRK'
        elif country_name == 'Republic of Korea': # South Korea
            return 'KOR'
        else:
            return countries.get(country_name).alpha3
    except KeyError:
        print(f"Warning: Could not find ISO code for '{country_name}'")
        return None

df_map['ISO_Alpha3'] = df_map['Country'].apply(get_iso_alpha3)

# Drop rows where ISO code could not be determined
df_map.dropna(subset=['ISO_Alpha3'], inplace=True)

# Count the number of launches per country
launches_by_country = df_map['ISO_Alpha3'].value_counts().reset_index()
launches_by_country.columns = ['ISO_Alpha3', 'Number of Launches']

# Create the choropleth map
fig = px.choropleth(launches_by_country,
                    locations='ISO_Alpha3',
                    color='Number of Launches',
                    hover_name='ISO_Alpha3',
                    color_continuous_scale=px.colors.sequential.matter,
                    title='Number of Space Mission Launches by Country')

fig.update_layout(coloraxis_colorbar=dict(
    title='Number of Launches',
    tickvals=[launches_by_country['Number of Launches'].min(), launches_by_country['Number of Launches'].max()],
    ticktext=[str(launches_by_country['Number of Launches'].min()), str(launches_by_country['Number of Launches'].max())]
))

fig.show()

# Use a Choropleth Map to Show the Number of Failures by Country


In [16]:
failed_missions = df_map[df_map['Mission_Status'].isin(['Failure', 'Partial Failure', 'Prelaunch Failure'])]

# Count the number of failures per country
failures_by_country = failed_missions['ISO_Alpha3'].value_counts().reset_index()
failures_by_country.columns = ['ISO_Alpha3', 'Number of Failures']

# Create the choropleth map for failures
fig_failures = px.choropleth(failures_by_country,
                             locations='ISO_Alpha3',
                             color='Number of Failures',
                             hover_name='ISO_Alpha3',
                             color_continuous_scale=px.colors.sequential.matter,
                             title='Number of Space Mission Failures by Country')

fig_failures.update_layout(coloraxis_colorbar=dict(
    title='Number of Failures',
    tickvals=[failures_by_country['Number of Failures'].min(), failures_by_country['Number of Failures'].max()],
    ticktext=[str(failures_by_country['Number of Failures'].min()), str(failures_by_country['Number of Failures'].max())]
))

fig_failures.show()

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status.

In [17]:
sunburst_data = df_map.groupby(['Country', 'Organisation', 'Mission_Status']).size().reset_index(name='Number of Launches')

fig_sunburst = px.sunburst(sunburst_data,
                           path=['Country', 'Organisation', 'Mission_Status'],
                           values='Number of Launches',
                           title='Space Mission Launches: Country > Organisation > Mission Status',
                           color_discrete_sequence=px.colors.sequential.Plasma)

fig_sunburst.show()

# Analyse the Total Amount of Money Spent by Organisation on Space Missions

In [21]:
df_data['Price'] = pd.to_numeric(df_data['Price'], errors='coerce')
money_spent_by_organisation = df_data.dropna(subset=['Price']).groupby('Organisation')['Price'].sum().reset_index()
money_spent_by_organisation.columns = ['Organisation', 'Total Price (USD Millions)']
money_spent_by_organisation = money_spent_by_organisation.sort_values(by='Total Price (USD Millions)', ascending=False)

print("Total Amount of Money Spent by Organisation on Space Missions (USD Millions):")
print(money_spent_by_organisation)

Total Amount of Money Spent by Organisation on Space Missions (USD Millions):
       Organisation  Total Price (USD Millions)
14             NASA                   61,200.00
0       Arianespace                   16,345.00
20              ULA                   14,798.00
2              CASC                    6,340.26
19           SpaceX                    5,444.00
15         Northrop                    3,930.00
12              MHI                    3,532.50
8              ISRO                    2,177.00
21     US Air Force                    1,550.92
22           VKS RF                    1,548.90
7               ILS                    1,320.00
1            Boeing                    1,241.00
17        Roscosmos                    1,187.50
13  Martin Marietta                      721.40
10        Kosmotras                      638.00
5          Eurockot                      543.40
11         Lockheed                      280.00
9              JAXA                      168.00
16       R

# Chart the Number of Launches per Year

In [23]:
df_data['Date'] = pd.to_datetime(df_data['Date'], format='mixed', errors='coerce', utc=True)
df_data['Year'] = df_data['Date'].dt.year

launches_per_year = df_data['Year'].value_counts().sort_index().reset_index()
launches_per_year.columns = ['Year', 'Number of Launches']

fig_year = px.bar(launches_per_year,
                   x='Year',
                   y='Number of Launches',
                   title='Number of Space Mission Launches Per Year',
                   labels={'Year': 'Year', 'Number of Launches': 'Number of Launches'},
                   color_discrete_sequence=px.colors.sequential.Viridis)

fig_year.update_layout(xaxis_title='Year', yaxis_title='Number of Launches')
fig_year.show()

# Chart the Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart.

In [24]:
df_monthly_launches = df_data.dropna(subset=['Date']).copy()
df_monthly_launches['YearMonth'] = df_monthly_launches['Date'].dt.to_period('M')

monthly_launches_count = df_monthly_launches['YearMonth'].value_counts().sort_index().reset_index()
monthly_launches_count.columns = ['YearMonth', 'Number of Launches']
monthly_launches_count['YearMonth'] = monthly_launches_count['YearMonth'].dt.to_timestamp()

# Calculate a 12-month rolling average
monthly_launches_count['Rolling_Average'] = monthly_launches_count['Number of Launches'].rolling(window=12, center=True).mean()

# Create the time series chart with rolling average
fig_month_on_month = px.line(monthly_launches_count,
                             x='YearMonth',
                             y='Number of Launches',
                             title='Number of Space Mission Launches Month-on-Month with Rolling Average',
                             labels={'YearMonth': 'Date', 'Number of Launches': 'Number of Launches'})

fig_month_on_month.add_scatter(x=monthly_launches_count['YearMonth'], y=monthly_launches_count['Rolling_Average'], mode='lines', name='12-Month Rolling Average', line=dict(color='red', width=2))

fig_month_on_month.update_layout(xaxis_title='Date', yaxis_title='Number of Launches')
fig_month_on_month.show()

# Find the month with the highest number of launches in all time (across all years)
df_data['Month'] = df_data['Date'].dt.month
launches_by_month_overall = df_data['Month'].value_counts().sort_index()

# Map month number to month name for better readability
month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
               7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
launches_by_month_overall.index = launches_by_month_overall.index.map(month_names)

highest_month = launches_by_month_overall.idxmax()
highest_month_count = launches_by_month_overall.max()

print(f"\nMonth with the highest number of launches in all time: {highest_month} with {highest_month_count} launches.")

/tmp/ipykernel_10532/3398953562.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_monthly_launches['YearMonth'] = df_monthly_launches['Date'].dt.to_period('M')



Month with the highest number of launches in all time: December with 450 launches.


# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [25]:
launches_by_month = df_data['Month'].value_counts().sort_index().reset_index()
launches_by_month.columns = ['Month_Number', 'Number of Launches']

# Map month number to month name for better readability
month_names_full = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
                    7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
launches_by_month['Month'] = launches_by_month['Month_Number'].map(month_names_full)

# Create the bar chart
fig_month_popularity = px.bar(launches_by_month,
                               x='Month',
                               y='Number of Launches',
                               title='Number of Space Mission Launches per Month (All Years)',
                               labels={'Month': 'Month', 'Number of Launches': 'Number of Launches'},
                               color='Number of Launches',
                               color_continuous_scale=px.colors.sequential.Viridis)

fig_month_popularity.update_layout(xaxis_title='Month', yaxis_title='Number of Launches')
fig_month_popularity.show()

# Identify the most and least popular months
most_popular_month_name = launches_by_month.loc[launches_by_month['Number of Launches'].idxmax(), 'Month']
most_popular_month_count = launches_by_month['Number of Launches'].max()

least_popular_month_name = launches_by_month.loc[launches_by_month['Number of Launches'].idxmin(), 'Month']
least_popular_month_count = launches_by_month['Number of Launches'].min()

print(f"\nMost popular month for launches: {most_popular_month_name} with {most_popular_month_count} launches.")
print(f"Least popular month for launches: {least_popular_month_name} with {least_popular_month_count} launches.")


Most popular month for launches: December with 450 launches.
Least popular month for launches: January with 268 launches.


# How has the Launch Price varied Over Time?

Create a line chart that shows the average price of rocket launches over time.

In [26]:
average_price_per_year = df_data.dropna(subset=['Price']).groupby('Year')['Price'].mean().reset_index()
average_price_per_year.columns = ['Year', 'Average Price (USD Millions)']

fig_price_over_time = px.line(average_price_per_year,
                              x='Year',
                              y='Average Price (USD Millions)',
                              title='Average Price of Rocket Launches Over Time (USD Millions)',
                              labels={'Year': 'Year', 'Average Price (USD Millions)': 'Average Price (USD Millions)'},
                              markers=True)

fig_price_over_time.update_layout(xaxis_title='Year', yaxis_title='Average Price (USD Millions)')
fig_price_over_time.show()

# Chart the Number of Launches over Time by the Top 10 Organisations.

How has the dominance of launches changed over time between the different players?

In [27]:
top_10_organisations = df_data['Organisation'].value_counts().head(10).index

df_top_organisations = df_data[df_data['Organisation'].isin(top_10_organisations)].copy()

launches_by_org_year = df_top_organisations.groupby(['Year', 'Organisation']).size().reset_index(name='Number of Launches')

fig_org_dominance = px.line(launches_by_org_year,
                            x='Year',
                            y='Number of Launches',
                            color='Organisation',
                            title='Number of Launches Over Time by Top 10 Organisations',
                            labels={'Year': 'Year', 'Number of Launches': 'Number of Launches'},
                            hover_name='Organisation')

fig_org_dominance.update_layout(xaxis_title='Year', yaxis_title='Number of Launches')
fig_org_dominance.show()

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991.

In [28]:
# Filter data for the Cold War period (up to and including 1991)
df_cold_war = df_data[df_data['Year'] <= 1991].copy()

# Identify USSR and USA organisations
us_organisations = ['NASA', 'US Air Force', 'General Dynamics', 'Boeing', 'Lockheed', 'Martin Marietta', 'McDonnell Douglas', 'Sandia', 'Vought']
ussr_organisations = ['RVSN USSR', 'VKS RF', 'Roscosmos', 'Kosmotras', 'Khrunichev']

# Filter for only USA and USSR relevant organisations
df_cold_war_usa_ussr = df_cold_war[df_cold_war['Organisation'].isin(us_organisations + ussr_organisations)].copy()

print(f"Cold War dataset shape: {df_cold_war_usa_ussr.shape}")

Cold War dataset shape: (2401, 9)


## Create a Plotly Pie Chart comparing the total number of launches of the USSR and the USA

Hint: Remember to include former Soviet Republics like Kazakhstan when analysing the total number of launches.

In [29]:
# Assign 'Country_Group' based on identified organisations
def assign_country_group(organisation):
    if organisation in us_organisations:
        return 'USA'
    elif organisation in ussr_organisations:
        return 'USSR'
    else:
        return 'Other'

df_cold_war_usa_ussr['Country_Group'] = df_cold_war_usa_ussr['Organisation'].apply(assign_country_group)

# Filter out 'Other' if any organisations outside USA/USSR were included in df_cold_war_usa_ussr
df_cold_war_usa_ussr_filtered = df_cold_war_usa_ussr[df_cold_war_usa_ussr['Country_Group'].isin(['USA', 'USSR'])]

launches_by_superpower = df_cold_war_usa_ussr_filtered['Country_Group'].value_counts().reset_index()
launches_by_superpower.columns = ['Superpower', 'Number of Launches']

fig_pie_chart = px.pie(launches_by_superpower,
                       values='Number of Launches',
                       names='Superpower',
                       title='Total Number of Launches: USA vs USSR (Cold War Era)',
                       color_discrete_sequence=px.colors.qualitative.Pastel)

fig_pie_chart.update_traces(textposition='inside', textinfo='percent+label')
fig_pie_chart.show()

print("Total Launches during Cold War (USA vs USSR):")
print(launches_by_superpower)

Total Launches during Cold War (USA vs USSR):
  Superpower  Number of Launches
0       USSR                1766
1        USA                 635


## Create a Chart that Shows the Total Number of Launches Year-On-Year by the Two Superpowers

In [30]:
launches_by_superpower_year = df_cold_war_usa_ussr_filtered.groupby(['Year', 'Country_Group']).size().reset_index(name='Number of Launches')

fig_superpower_year = px.line(
    launches_by_superpower_year,
    x='Year',
    y='Number of Launches',
    color='Country_Group',
    title='Total Number of Launches Year-on-Year: USA vs USSR (Cold War Era)',
    labels={'Year': 'Year', 'Number of Launches': 'Number of Launches', 'Country_Group': 'Superpower'},
    markers=True,
    color_discrete_map={'USA': 'blue', 'USSR': 'red'}
)

fig_superpower_year.update_layout(xaxis_title='Year', yaxis_title='Number of Launches')
fig_superpower_year.show()

## Chart the Total Number of Mission Failures Year on Year.

In [31]:
failed_missions_year_on_year = df_data[df_data['Mission_Status'].isin(['Failure', 'Partial Failure', 'Prelaunch Failure'])].groupby('Year').size().reset_index(name='Number of Failures')

fig_failures_year_on_year = px.bar(
    failed_missions_year_on_year,
    x='Year',
    y='Number of Failures',
    title='Total Number of Mission Failures Year-on-Year',
    labels={'Year': 'Year', 'Number of Failures': 'Number of Failures'},
    color_discrete_sequence=px.colors.sequential.Reds
)

fig_failures_year_on_year.update_layout(xaxis_title='Year', yaxis_title='Number of Failures')
fig_failures_year_on_year.show()

## Chart the Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time?

In [32]:
total_launches_per_year = df_data.groupby('Year').size().reset_index(name='Total Launches')

failed_launches_per_year = df_data[df_data['Mission_Status'].isin(['Failure', 'Partial Failure', 'Prelaunch Failure'])] \
                                .groupby('Year').size().reset_index(name='Failed Launches')

# Merge the two dataframes
failure_percentage = pd.merge(total_launches_per_year, failed_launches_per_year, on='Year', how='left')
failure_percentage['Failed Launches'] = failure_percentage['Failed Launches'].fillna(0) # Fill NaN with 0 for years with no failures

failure_percentage['Percentage Failures'] = (failure_percentage['Failed Launches'] / failure_percentage['Total Launches']) * 100

fig_failure_percentage = px.line(
    failure_percentage,
    x='Year',
    y='Percentage Failures',
    title='Percentage of Mission Failures Over Time',
    labels={'Year': 'Year', 'Percentage Failures': 'Percentage of Failures (%)'},
    markers=True,
    color_discrete_sequence=px.colors.sequential.Sunset
)

fig_failure_percentage.update_layout(xaxis_title='Year', yaxis_title='Percentage of Failures (%)')
fig_failure_percentage.show()

# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020)

Do the results change if we only look at the number of successful launches?

In [34]:
# Ensure 'Date' column in df_map is datetime and extract 'Year'
df_map['Date'] = pd.to_datetime(df_map['Date'], format='mixed', errors='coerce', utc=True)
df_map['Year'] = df_map['Date'].dt.year

# Filter df_map for years up to and including 2020
df_filtered_2020 = df_map[df_map['Year'] <= 2020].copy()

# --- Analysis for Total Launches ---
print("\n--- Country Lead by Total Number of Launches (Year-on-Year until 2020) ---")

total_launches_by_country_year = df_filtered_2020.groupby(['Year', 'Country']).size().reset_index(name='Total Launches')

# Find the country with the maximum launches for each year
idx_max_total = total_launches_by_country_year.groupby('Year')['Total Launches'].idxmax()
lead_country_total = total_launches_by_country_year.loc[idx_max_total].reset_index(drop=True)

print(lead_country_total)

# --- Analysis for Successful Launches Only ---
print("\n--- Country Lead by Total Number of SUCCESSFUL Launches (Year-on-Year until 2020) ---")

successful_launches_by_country_year = df_filtered_2020[
    df_filtered_2020['Mission_Status'] == 'Success'
].groupby(['Year', 'Country']).size().reset_index(name='Successful Launches')

# Find the country with the maximum successful launches for each year
idx_max_success = successful_launches_by_country_year.groupby('Year')['Successful Launches'].idxmax()
lead_country_success = successful_launches_by_country_year.loc[idx_max_success].reset_index(drop=True)

print(lead_country_success)

# --- Comparison ---
print("\n--- Comparison of Leads ---")
merged_leads = pd.merge(lead_country_total, lead_country_success, on='Year', suffixes=('_Total', '_Successful'), how='outer')
merged_leads['Lead Change'] = merged_leads['Country_Total'] != merged_leads['Country_Successful']

print("Summary of Lead Changes:")
print(merged_leads[merged_leads['Lead Change']].to_string())

if merged_leads['Lead Change'].any():
    print("\nYes, the results do change when only looking at the number of successful launches for some years.")
    print("This indicates that while a country might have had the highest total launches in a given year, another country might have had more successful launches, or the same country that leads in total launches might not always have the highest success rate.")
else:
    print("\nNo, the results do not change significantly when only looking at the number of successful launches. The same countries consistently lead in both total and successful missions.")


--- Country Lead by Total Number of Launches (Year-on-Year until 2020) ---
    Year     Country  Total Launches
0   1957  Kazakhstan               2
1   1958         USA              23
2   1959         USA              16
3   1960         USA              30
4   1961         USA              43
..   ...         ...             ...
59  2016         USA              27
60  2017         USA              30
61  2018       China              39
62  2019       China              34
63  2020       China              22

[64 rows x 3 columns]

--- Country Lead by Total Number of SUCCESSFUL Launches (Year-on-Year until 2020) ---
    Year     Country  Successful Launches
0   1957  Kazakhstan                    2
1   1958         USA                    5
2   1959         USA                    6
3   1960         USA                   16
4   1961         USA                   27
..   ...         ...                  ...
59  2016         USA                   26
60  2017         USA              

# Create a Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

Which organisation was dominant in the 1970s and 1980s? Which organisation was dominant in 2018, 2019 and 2020?

In [35]:
# Group by Year and Organisation to count launches
launches_by_year_org = df_data.groupby(['Year', 'Organisation']).size().reset_index(name='Number of Launches')

# Find the organisation with the most launches for each year
idx_max_launches = launches_by_year_org.groupby('Year')['Number of Launches'].idxmax()
dominant_organisation_year = launches_by_year_org.loc[idx_max_launches].reset_index(drop=True)

# Create the bar chart for dominant organisations year-on-year
fig_dominant_org_year = px.bar(
    dominant_organisation_year,
    x='Year',
    y='Number of Launches',
    color='Organisation',
    title='Organisation with Most Launches Year-on-Year',
    labels={'Year': 'Year', 'Number of Launches': 'Number of Launches', 'Organisation': 'Dominant Organisation'},
    hover_name='Organisation',
    color_discrete_sequence=px.colors.qualitative.Plotly
)

fig_dominant_org_year.update_layout(xaxis_title='Year', yaxis_title='Number of Launches')
fig_dominant_org_year.show()

print("\n--- Dominant Organisations Analysis ---")

# Dominant in 1970s (1970-1979)
dominant_70s = dominant_organisation_year[
    (dominant_organisation_year['Year'] >= 1970) & (dominant_organisation_year['Year'] <= 1979)
]
print("Dominant organisation(s) in the 1970s:")
print(dominant_70s.groupby('Organisation')['Number of Launches'].sum().sort_values(ascending=False).head(1))

# Dominant in 1980s (1980-1989)
dominant_80s = dominant_organisation_year[
    (dominant_organisation_year['Year'] >= 1980) & (dominant_organisation_year['Year'] <= 1989)
]
print("\nDominant organisation(s) in the 1980s:")
print(dominant_80s.groupby('Organisation')['Number of Launches'].sum().sort_values(ascending=False).head(1))

# Dominant in 2018, 2019, 2020
dominant_recent = dominant_organisation_year[
    dominant_organisation_year['Year'].isin([2018, 2019, 2020])
]
print("\nDominant organisation(s) in 2018, 2019, and 2020:")
print(dominant_recent)


--- Dominant Organisations Analysis ---
Dominant organisation(s) in the 1970s:
Organisation
RVSN USSR    814
Name: Number of Launches, dtype: int64

Dominant organisation(s) in the 1980s:
Organisation
RVSN USSR    436
Name: Number of Launches, dtype: int64

Dominant organisation(s) in 2018, 2019, and 2020:
    Year Organisation  Number of Launches
61  2018         CASC                  37
62  2019         CASC                  27
63  2020         CASC                  19
